## Импорт Датасетов

In [845]:
#!pip install kagglehub

Поменяем переменную среды чтобы скачать датасет куда нам надо

In [846]:
import os
os.environ['KAGGLEHUB_CACHE'] = "C:/Users/alesh/PycharmProjects/PythonProject/data"

In [847]:
import kagglehub
KAGGLEHUB_CACHE = "C:/Users/alesh/PycharmProjects/PythonProject/data"

path = kagglehub.dataset_download(
    "uciml/iris",
    path="Iris.csv"
)

print("Path to dataset files:", path)

Path to dataset files: C:/Users/alesh/PycharmProjects/PythonProject/data\datasets\uciml\iris\versions\2\Iris.csv


Экспорт в переменную

In [848]:
import pandas as pd

train = pd.read_csv(path)

train.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## Создание Нейросети

In [849]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

torch.manual_seed(42) # сразу задаём seed

In [850]:
X = train[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']].values
y = train['Species'].values

Преобразуем тектовые метки в числовые

In [851]:
y = LabelEncoder().fit_transform(y)
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

Разделяем на обучающую и тестовую выборки

In [852]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y #  stratify сохраняет распределение классов
)

Переводим в тензоры PyTorch

In [853]:
X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

Создаём Dataset

In [854]:
class IrisDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return len(self.features)

train_dataset = IrisDataset(X_train, y_train)
test_dataset = IrisDataset(X_test, y_test)

Создаём DataLoader(создаем итераторы для батчевой загрузки данных)

In [855]:
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True) # shuffle=True перемешивает данные при обучении
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False) # batch_size=10 - модель видит 10 примеров за одну итерацию

Создание модели нейронной сети

In [856]:
class IrisNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layers = nn.Sequential(
            # 1st layer
            nn.Linear(input_size, hidden_size),  # Полносвязный слой 1
            nn.ReLU(), # активационная функция

            # 2nd layer
            nn.Linear(hidden_size, hidden_size), # Полносвязный слой 2
            nn.ReLU(),

            # 3nd layer
            nn.Linear(hidden_size, hidden_size), # Полносвязный слой 3
            nn.ReLU(),

            # 4 layer
            nn.Linear(hidden_size, output_size) # Выходной слой
        )

    # прогоняем вектор x через все слои
    def forward(self, features):
        return self.layers(features) # возвращает "сырое" число принадлежности классу

model = IrisNet(input_size=4, hidden_size=10, output_size=3)

Определение оптимизатора

In [857]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01) # алгоритм, который обновляет веса модели, чтобы минимизировать функцию потерь

Обучение модели

In [858]:
num_epochs = 200 # количество эпох обучения (полных проходов через весь набор данных)
for epoch in range(num_epochs):
    model.train()  # Режим обучения
    total_loss = 0

# текущий батч,тензор с признаками батча, тензор с правильными ответами
# enumerate() добавляет счётчик к итерации
# каждая итерация train_loader возвращает один батч данных
    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.float()

        # Прямой проход
        outputs = model(data) # вызов forward
        loss = F.cross_entropy(outputs, target) # показывает в процентах насколько отличается первый тензор от второго

        # Обратный проход
        # Оптимизатор двигает параметры в направлении, противоположном градиенту (чтобы УМЕНЬШИТЬ потери)
        optimizer.zero_grad()  # Обнуляем градиенты; чтобы не копились
        loss.backward()        # Вычисляем градиенты; для новой итерации
        optimizer.step()       # Обновляем веса;

        total_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}')

Epoch [10/200], Loss: 1.0965
Epoch [20/200], Loss: 1.0663
Epoch [30/200], Loss: 1.0077
Epoch [40/200], Loss: 0.8161
Epoch [50/200], Loss: 0.5649
Epoch [60/200], Loss: 0.4118
Epoch [70/200], Loss: 0.3115
Epoch [80/200], Loss: 0.2559
Epoch [90/200], Loss: 0.1898
Epoch [100/200], Loss: 0.1630
Epoch [110/200], Loss: 0.1912
Epoch [120/200], Loss: 0.1156
Epoch [130/200], Loss: 0.1187
Epoch [140/200], Loss: 0.1001
Epoch [150/200], Loss: 0.1136
Epoch [160/200], Loss: 0.0959
Epoch [170/200], Loss: 0.0948
Epoch [180/200], Loss: 0.1649
Epoch [190/200], Loss: 0.0830
Epoch [200/200], Loss: 0.0788


Оценка модели

In [871]:
model.eval()  # Режим оценки
correct = 0
total = 0

with torch.no_grad():  # Отключаем вычисление градиентов
    for data, target in test_loader:
        data = data.float()

        outputs = model(data)
        _, predicted = torch.max(outputs.data, dim=1)  # torch.max(tensor, dim) -> (values, indices); dim=1 - максимум по второму измерению(по классам)
        total += target.size(0) # возвращает размер батча
        correct += (predicted == target).sum()

print(f'Accuracy: {100 * correct / total:.2f}%')

Accuracy: 100.00%
